In [1]:
# -*- coding: utf-8 -*-
"""
Combined DNN+GNN model for high-energy physics classification.

This script loads pre-processed event data (leptons, jets, fat jets, MET)
and fat jet constituent data.

It uses a GNN (Interaction Network) to process the constituents of the
leading fat jet, creating a learned representation. This representation
is then combined with high-level event features (lepton kinematics,
other jet kinematics, MET) and fed into a Deep Neural Network (DNN)
for binary classification (signal vs. background).
"""

"""
## Setup
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

# --- Plot Helpers (from original script) ---

def plot_histogram(df, column, bins=50, title=None, xlabel=None, ylabel="Frequency", yscale ='linear'):
    if column not in df.columns:
        raise ValueError(f"Column '{column}' not found in DataFrame.")

    plt.figure()
    plt.hist(df[column], bins=bins, edgecolor='black')
    plt.title(title or f"Histogram of {column}")
    plt.xlabel(xlabel or column)
    plt.ylabel(ylabel)
    plt.yscale(yscale)
    plt.grid(True)
    plt.show()

def plot_histogram_by_type(df, value_column, type_column="type", bins=50, alpha=0.7, filename = 'default_file', title = None):
    if value_column not in df.columns or type_column not in df.columns:
        raise ValueError(f"Columns '{value_column}' and/or '{type_column}' not found in DataFrame.")

    # Compute shared bin edges from entire column
    data = df[value_column].dropna()
    bin_edges = np.linspace(data.min(), data.max(), bins + 1)

    plt.figure(figsize=(8, 6))

    for t in df[type_column].unique():
        subset = df[df[type_column] == t][value_column].dropna()
        plt.hist(subset, bins=bin_edges, alpha=alpha, label=t, edgecolor='black', histtype='stepfilled')

    plt.xlabel(value_column)
    plt.ylabel("Frequency")
    plt.yscale('log')
    title = title or f"Histogram of {value_column} by {type_column} from {filename}"
    plt.title(title)
    plt.legend(title=type_column)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# --- Physics Helper Functions (from original script) ---

def delta_phi(phi1, phi2):
    """Computes the difference in phi, handling the wrap-around at pi."""
    # Ensure inputs are numpy arrays for broadcasting
    dphi = np.asarray(phi1) - np.asarray(phi2)
    dphi = np.where(dphi > np.pi, dphi - 2 * np.pi, dphi)
    dphi = np.where(dphi < -np.pi, dphi + 2 * np.pi, dphi)
    return dphi

def delta_r(eta1, eta2, phi1, phi2):
    """Computes the delta R distance between two objects."""
    deta = np.asarray(eta1) - np.asarray(eta2)
    dphi = delta_phi(phi1, phi2)
    return np.sqrt(deta**2 + dphi**2)



ModuleNotFoundError: No module named 'seaborn'

In [5]:
path_name = "data/"


In [3]:

# Signal
signal_df_lep = pd.read_csv(path_name + "signal_df_lep.csv")
signal_df_g = pd.read_csv(path_name + "signal_df_g.csv")
signal_df_j = pd.read_csv(path_name + "signal_df_j_cleaned.csv")
signal_df_fj = pd.read_csv(path_name + "signal_df_fj_cleaned.csv")
signal_df_met = pd.read_csv(path_name + "signal_df_met.csv")
signal_df_fjc = pd.read_csv(path_name + "signal_df_fjc.csv")






In [ ]:
# Background 500
background_500_df_lep = pd.read_csv(path_name + "background_500_df_lep.csv")
background_500_df_j = pd.read_csv(path_name + "background_500_df_j_cleaned.csv")
background_500_df_fj = pd.read_csv(path_name + "background_500_df_fj_cleaned.csv")
background_500_df_met = pd.read_csv(path_name + "background_500_df_met.csv")
background_500_df_fjc = pd.read_csv(path_name + "background_500_df_fjc.csv")

# Background 1000
background_1000_df_lep = pd.read_csv(path_name + "background_1000_df_lep.csv")
background_1000_df_j = pd.read_csv(path_name + "background_1000_df_j_cleaned.csv")
background_1000_df_fj = pd.read_csv(path_name + "background_1000_df_fj_cleaned.csv")
background_1000_df_met = pd.read_csv(path_name + "background_1000_df_met.csv")
background_1000_df_fjc = pd.read_csv(path_name + "background_1000_df_fjc.csv")

In [6]:
# Background 280
background_280_df_lep = pd.read_csv(path_name + "background_280_df_lep.csv")
background_280_df_j = pd.read_csv(path_name + "background_280_df_j_cleaned.csv")
background_280_df_fj = pd.read_csv(path_name + "background_280_df_fj_cleaned.csv")
background_280_df_met = pd.read_csv(path_name + "background_280_df_met.csv")
background_280_df_fjc = pd.read_csv(path_name + "background_280_df_fjc.csv")

In [7]:
# -*- coding: utf-8 -*-
"""
Step 1: Physics Selection & Filtering (Parquet)

Logic:
1. Identify valid events (Must have lepton + fat jet).
2. Select exactly ONE fat jet per event (closest to lepton).
3. Filter all other dataframes to these events.
4. Save as Parquet in original 'long' format (no pivoting yet).
"""

import os
import numpy as np
import pandas as pd

# --- Setup Output Directory ---
OUTPUT_DIR = 'data_filtered'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Saving processed files to: {OUTPUT_DIR}/")

# --- Helper Functions ---

def delta_phi(phi1, phi2):
    dphi = np.asarray(phi1) - np.asarray(phi2)
    dphi = np.where(dphi > np.pi, dphi - 2 * np.pi, dphi)
    dphi = np.where(dphi < -np.pi, dphi + 2 * np.pi, dphi)
    return dphi

def delta_r(eta1, eta2, phi1, phi2):
    deta = np.asarray(eta1) - np.asarray(eta2)
    dphi = delta_phi(phi1, phi2)
    return np.sqrt(deta**2 + dphi**2)

def filter_and_save(name, df_lep, df_j, df_fj, df_met, df_fjc, df_gen=None):
    print(f"\nProcessing {name}...")
    
    # 1. Identify Common Valid Events
    # Must have at least 1 lepton and 1 fat jet
    valid_lep_idx = df_lep['event_index'].unique()
    valid_fj_idx = df_fj['event_index'].unique()
    
    # Intersection
    valid_events = np.intersect1d(valid_lep_idx, valid_fj_idx)
    print(f"  - Original events: {len(valid_lep_idx)} (lep) / {len(valid_fj_idx)} (fj)")
    print(f"  - Valid intersection: {len(valid_events)}")

    # 2. Filter Lepton (Keep all valid events, leading lepton only usually desired, 
    # but user said "drop bad events", implying we keep the structure. 
    # We'll filter to valid events. If there are multiple leptons, we keep them 
    # (unless you strictly want 1). Assuming 1 per event based on previous code.)
    df_lep_filtered = df_lep[df_lep['event_index'].isin(valid_events)].copy()
    
    # To do the dR calculation, we need the leading lepton info temporarily
    # Sort by pT desc to get leading lepton
    df_lep_lead = df_lep_filtered.sort_values(['event_index', 'pt'], ascending=[True, False])
    df_lep_lead = df_lep_lead.groupby('event_index').head(1).set_index('event_index')[['eta', 'phi']]
    df_lep_lead = df_lep_lead.rename(columns={'eta': 'l_eta', 'phi': 'l_phi'})

    # 3. Select ONE Fat Jet (Closest to Lepton)
    df_fj_filtered = df_fj[df_fj['event_index'].isin(valid_events)].copy()
    
    # Merge lepton info
    df_fj_calc = df_fj_filtered.merge(df_lep_lead, left_on='event_index', right_index=True)
    
    # Calculate dR
    df_fj_calc['dr_lep'] = delta_r(df_fj_calc['eta'], df_fj_calc['l_eta'], 
                                   df_fj_calc['phi'], df_fj_calc['l_phi'])
    
    # Sort by dR ascending and take head(1)
    df_fj_selected = df_fj_calc.sort_values(['event_index', 'dr_lep']).groupby('event_index').head(1)
    
    # Drop the temp calc columns to restore original format
    df_fj_final = df_fj_selected.drop(columns=['l_eta', 'l_phi', 'dr_lep'])
    
    # 4. Filter Constituents (_fjc)
    # We only want constituents that belong to the selected fat jet.
    # Inner join on event_index AND 'ind' (which links to df_fj 'index')
    
    # We assume df_fj has a column 'index' that maps to df_fjc['ind'].
    # If df_fj uses a different column name for its ID, update 'right_on'.
    if 'index' not in df_fj_final.columns:
        # Fallback if 'index' is missing: assume 'ind' in fjc refers to rank 0 if we selected rank 0?
        # But since we selected by dR, the rank varies. 
        # We MUST have the linking ID. Assuming input has 'index'.
        raise ValueError("df_fj missing 'index' column required to link constituents.")
        
    keys = df_fj_final[['event_index', 'index']]
    df_fjc_final = df_fjc.merge(keys, left_on=['event_index', 'ind'], right_on=['event_index', 'index'], how='inner')
    
    # Clean up merge artifacts if any (pandas might duplicate keys if names match, but here they match)
    # Ensure we don't have duplicate columns
    df_fjc_final = df_fjc_final.loc[:, ~df_fjc_final.columns.duplicated()]

    # 5. Filter Jets (_j) and MET (_met)
    # Simply keep all rows corresponding to the valid events
    df_j_final = df_j[df_j['event_index'].isin(valid_events)].copy()
    df_met_final = df_met[df_met['event_index'].isin(valid_events)].copy()

    # 6. (Optional) Signal Specific: Higgs matching
    # If this is signal, we might want to verify the Higgs is inside the jet.
    # But the prompt says "drop bad events e.g. if a event does not have any lepton".
    # It didn't strictly say "drop events where Higgs isn't in jet" for this step.
    # However, to be consistent with previous logic, if you want that cut, apply it here.
    # I will skip the geometric Higgs cut for now to strictly follow "save as how they are read but filtered".
    # The ML model handles training cuts.
    
    # 7. Save to Parquet
    print(f"  - Saving {len(df_lep_filtered)} lepton rows")
    df_lep_filtered.to_parquet(f"{OUTPUT_DIR}/{name}_df_lep.parquet")
    
    print(f"  - Saving {len(df_j_final)} jet rows")
    df_j_final.to_parquet(f"{OUTPUT_DIR}/{name}_df_j.parquet")
    
    print(f"  - Saving {len(df_fj_final)} fat jet rows (1 per event)")
    df_fj_final.to_parquet(f"{OUTPUT_DIR}/{name}_df_fj.parquet")
    
    print(f"  - Saving {len(df_met_final)} MET rows")
    df_met_final.to_parquet(f"{OUTPUT_DIR}/{name}_df_met.parquet")
    
    print(f"  - Saving {len(df_fjc_final)} constituent rows")
    df_fjc_final.to_parquet(f"{OUTPUT_DIR}/{name}_df_fjc.parquet")



Saving processed files to: data_filtered/


In [ ]:
def filter_and_save_signal(df_lep, df_j, df_fj, df_met, df_fjc, df_g):
    print("\nProcessing Signal (with Higgs Truth Matching)...")
    
    # 1. Identify Common Valid Events (Lepton + Fat Jet)
    valid_lep_idx = df_lep['event_index'].unique()
    valid_fj_idx = df_fj['event_index'].unique()
    valid_events = np.intersect1d(valid_lep_idx, valid_fj_idx)
    print(f"  - Initial valid events (Lep+FJ): {len(valid_events)}")

    # 2. Select ONE Fat Jet (Closest to Lepton)
    # We first filter FJ and Lep to the valid list to speed up calc
    df_fj_filtered = df_fj[df_fj['event_index'].isin(valid_events)].copy()
    df_lep_filtered = df_lep[df_lep['event_index'].isin(valid_events)].copy()
    
    # Get leading lepton coordinates
    df_lep_lead = df_lep_filtered.sort_values(['event_index', 'pt'], ascending=[True, False])
    df_lep_lead = df_lep_lead.groupby('event_index').head(1).set_index('event_index')[['eta', 'phi']]
    df_lep_lead = df_lep_lead.rename(columns={'eta': 'l_eta', 'phi': 'l_phi'})

    # Merge lepton info onto fat jets
    df_fj_calc = df_fj_filtered.merge(df_lep_lead, left_on='event_index', right_index=True)
    
    # Calculate dR(lep, fj)
    df_fj_calc['dr_lep'] = delta_r(df_fj_calc['eta'], df_fj_calc['l_eta'], 
                                   df_fj_calc['phi'], df_fj_calc['l_phi'])
    
    # Select closest fat jet
    df_fj_selected = df_fj_calc.sort_values(['event_index', 'dr_lep']).groupby('event_index').head(1)
    
    # 3. [CRITICAL] Higgs Truth Matching
    # Get True Higgs coordinates from df_g (id == 25)
    # We assume there is one Higgs per event in the signal file
    df_higgs = df_g[df_g['id'] == 25][['event_index', 'eta', 'phi']].set_index('event_index')
    df_higgs = df_higgs.rename(columns={'eta': 'h_eta', 'phi': 'h_phi'})
    
    # Merge Higgs info onto the Selected Fat Jet
    # Note: 'inner' join here will also drop events that somehow don't have a Higgs record
    df_fj_matched = df_fj_selected.merge(df_higgs, left_on='event_index', right_index=True, how='inner')
    
    # Calculate dR(Higgs, FatJet)
    df_fj_matched['dr_hfj'] = delta_r(df_fj_matched['eta'], df_fj_matched['h_eta'], 
                                      df_fj_matched['phi'], df_fj_matched['h_phi'])
    
    # FILTER: Keep only events where the selected jet is close to the Higgs
    df_fj_final = df_fj_matched[df_fj_matched['dr_hfj'] < 1.0].copy()
    
    # Get the final list of verified signal events
    final_signal_events = df_fj_final['event_index'].unique()
    print(f"  - Events passing Higgs Matching (dR < 1.0): {len(final_signal_events)}")
    
    # Clean up the fat jet dataframe (drop calc columns)
    df_fj_final = df_fj_final.drop(columns=['l_eta', 'l_phi', 'dr_lep', 'h_eta', 'h_phi', 'dr_hfj'])

    # 4. Filter All Other Dataframes to this Final Event List
    df_lep_final = df_lep[df_lep['event_index'].isin(final_signal_events)].copy()
    df_j_final = df_j[df_j['event_index'].isin(final_signal_events)].copy()
    df_met_final = df_met[df_met['event_index'].isin(final_signal_events)].copy()
    
    # 5. Filter Constituents
    keys = df_fj_final[['event_index', 'index']]
    df_fjc_final = df_fjc.merge(keys, left_on=['event_index', 'ind'], right_on=['event_index', 'index'], how='inner')
    df_fjc_final = df_fjc_final.loc[:, ~df_fjc_final.columns.duplicated()]

    # 6. Save to Parquet
    print(f"  - Saving {len(df_lep_final)} lepton rows")
    df_lep_final.to_parquet(f"{OUTPUT_DIR}/signal_df_lep.parquet")
    
    print(f"  - Saving {len(df_j_final)} jet rows")
    df_j_final.to_parquet(f"{OUTPUT_DIR}/signal_df_j.parquet")
    
    print(f"  - Saving {len(df_fj_final)} fat jet rows (1 per event)")
    df_fj_final.to_parquet(f"{OUTPUT_DIR}/signal_df_fj.parquet")
    
    print(f"  - Saving {len(df_met_final)} MET rows")
    df_met_final.to_parquet(f"{OUTPUT_DIR}/signal_df_met.parquet")
    
    print(f"  - Saving {len(df_fjc_final)} constituent rows")
    df_fjc_final.to_parquet(f"{OUTPUT_DIR}/signal_df_fjc.parquet")

In [ ]:
filter_and_save_signal(signal_df_lep, signal_df_j, signal_df_fj, signal_df_met, signal_df_fjc, signal_df_g)

In [ ]:

# ==========================================
# EXECUTION
# ==========================================

# 1. Signal
# Assuming DataFrames are loaded: signal_df_lep, etc.
# filter_and_save("signal", signal_df_lep, signal_df_j, signal_df_fj, signal_df_met, signal_df_fjc)

# 2. Backgrounds
# Assuming lists or separate DFs are loaded
backgrounds = [
    ("background_280", background_280_df_lep, background_280_df_j, background_280_df_fj, background_280_df_met, background_280_df_fjc),
    ("background_500", background_500_df_lep, background_500_df_j, background_500_df_fj, background_500_df_met, background_500_df_fjc),
    ("background_1000", background_1000_df_lep, background_1000_df_j, background_1000_df_fj, background_1000_df_met, background_1000_df_fjc),
]

for bg_name, bg_lep, bg_j, bg_fj, bg_met, bg_fjc in backgrounds:
    filter_and_save(bg_name, bg_lep, bg_j, bg_fj, bg_met, bg_fjc)

print("\nAll files filtered and saved to Parquet.")

In [ ]:

"""
# Model

## Event Processing (Phase 1: Data Prep)
"""

## Main Processing Function (MODIFIED)
def process_and_flatten_events(df_lep, df_j, df_fj, df_met, num_fat_jets_to_pick=1):
    """
    Prepares inputs for a machine learning model by flattening event data.

    MODIFIED: Also tracks the 'index' of the selected fat jet(s) to
    link with constituent data.
    """
    lep_features = ['pt', 'eta', 'phi', 'm', 'id']
    jet_features = ['pt', 'eta', 'phi', 'm']
    # --- MODIFIED: Added 'index' ---
    # 'index' from df_fj will be used to map to 'ind' in df_fjc
    fat_jet_features = ['pt', 'eta', 'phi', 'm', 'D2', 'count', 'index']
    met_features = ['pt', 'phi']

    # --- 0. Event Preselection ---
    jet_counts = df_j.groupby('event_index').size().rename('n_j')
    fat_jet_counts = df_fj.groupby('event_index').size().rename('n_fj')
    counts_df = pd.concat([jet_counts, fat_jet_counts], axis=1).fillna(0)
    valid_fat_jet_counts_idx = counts_df[counts_df['n_fj'] >= num_fat_jets_to_pick].index
    valid_lep_counts_idx = df_lep['event_index'].unique()
    valid_events_idx = np.intersect1d(valid_fat_jet_counts_idx, valid_lep_counts_idx)

    df_lep = df_lep[df_lep['event_index'].isin(valid_events_idx)]
    df_j = df_j[df_j['event_index'].isin(valid_events_idx)]
    df_fj = df_fj[df_fj['event_index'].isin(valid_events_idx)]
    df_met = df_met[df_met['event_index'].isin(valid_events_idx)]
    counts_df = counts_df.loc[valid_events_idx]

    # --- 1. Pre-process and Select Particles ---

    # Lepton
    df_lep = df_lep.sort_values(['event_index', 'pt'], ascending=[True, False])
    df_lep = df_lep.groupby('event_index').head(1)
    df_lep = df_lep.set_index('event_index')[lep_features].rename(columns={
        'pt': 'l_pt', 'eta': 'l_eta', 'phi': 'l_phi', 'm': 'l_m'
    })
    df_lep['l_charge'] = df_lep['id'] / abs(df_lep['id'])

    # MET
    df_met = df_met.set_index('event_index')[met_features].rename(columns={
        'pt': 'met_pt', 'phi': 'met_phi'
    })

    # Jets
    df_j = df_j.sort_values(['event_index', 'pt'], ascending=[True, False])
    df_j['jet_rank'] = df_j.groupby('event_index').cumcount()
    df_j_top2 = df_j[df_j['jet_rank'] < 2]
    wide_jets = df_j_top2.pivot(index='event_index', columns='jet_rank', values=jet_features)
    wide_jets.columns = [f'j{col[1]}_{col[0]}' for col in wide_jets.columns]
    for rank in range(2):
        for feature in jet_features:
            col_name = f'j{rank}_{feature}'
            if col_name not in wide_jets.columns:
                wide_jets[col_name] = 0
    wide_jets = wide_jets[[f'j{rank}_{feature}' for rank in range(2) for feature in jet_features]]

    # Fat Jets
    df_fj = df_fj.sort_values(['event_index', 'pt'], ascending=[True, False])
    df_fj['fat_jet_rank'] = df_fj.groupby('event_index').cumcount()
    df_fj = df_fj[df_fj['fat_jet_rank'] < num_fat_jets_to_pick]
    wide_fat_jets = df_fj.pivot(index='event_index', columns='fat_jet_rank', values=fat_jet_features)
    wide_fat_jets.columns = [f'fj{col[1]}_{col[0]}' for col in wide_fat_jets.columns]
    # This rename will now create 'fj_index'
    wide_fat_jets = wide_fat_jets.rename(columns=lambda c: c.replace('fj0_', 'fj_'))

    # --- 2. Merge All Information ---
    final_df = df_lep.join([counts_df, wide_jets, wide_fat_jets, df_met], how='inner')

    # --- 3. Calculate Derived Kinematic Features ---
    final_df['lj0_deta'] = np.where(final_df['j0_pt'] > 0, final_df['l_eta'] - final_df['j0_eta'], 0)
    final_df['lj0_dphi'] = np.where(final_df['j0_pt'] > 0, delta_phi(final_df['l_phi'], final_df['j0_phi']), 0)
    final_df['lj0_dr'] = np.where(final_df['j0_pt'] > 0, np.sqrt(final_df['lj0_deta']**2 + final_df['lj0_dphi']**2), 0)
    final_df['lj1_deta'] = np.where(final_df['j1_pt'] > 0, final_df['l_eta'] - final_df['j1_eta'], 0)
    final_df['lj1_dphi'] = np.where(final_df['j1_pt'] > 0, delta_phi(final_df['l_phi'], final_df['j1_phi']), 0)
    final_df['lj1_dr'] = np.where(final_df['j1_pt'] > 0, np.sqrt(final_df['lj1_deta']**2 + final_df['lj1_dphi']**2), 0)
    final_df['lfj_deta'] = final_df['l_eta'] - final_df['fj_eta']
    final_df['lfj_dphi'] = delta_phi(final_df['l_phi'], final_df['fj_phi'])
    final_df['lfj_dr'] = np.sqrt(final_df['lfj_deta']**2 + final_df['lfj_dphi']**2)
    final_df['lmet_dphi'] = delta_phi(final_df['l_phi'], final_df['met_phi'])
    final_df['fjmet_dphi'] = delta_phi(final_df['fj_phi'], final_df['met_phi'])

    # --- 4. Apply Additional Physics Criteria ---
    final_df = final_df[final_df['lfj_dr'] < 1].copy()

    # --- 5. Select Final Columns and Return ---
    # --- MODIFIED: Added 'fj_index' ---
    final_output_columns = [
        'l_pt', 'l_m', 'l_charge', 'l_eta',
        'j0_pt', 'j0_m',
        'j1_pt', 'j1_m',
        'fj_pt', 'fj_m', 'fj_D2', 'fj_count','fj_eta', 'fj_phi', 'fj_index', # Added fj_index
        'met_pt',
        'lj0_dr', 'lj0_dphi', 'lj0_deta',
        'lj1_dr', 'lj1_dphi', 'lj1_deta',
        'lfj_dr', 'lfj_dphi', 'lfj_deta',
        'lmet_dphi', 'fjmet_dphi',
        'n_fj', 'n_j'
    ]
    # We also need 'event_index' for the merge, so keep it until reset_index
    final_df = final_df.reindex(columns=final_output_columns)

    # Return with event_index for mapping
    return final_df.reset_index()

# --- NEW: Constituent Data Preparation Function ---
def prepare_constituent_data(df_fjc, df_fj, final_events_df,
                               max_constituents=50,
                               constituent_features=['pt', 'eta', 'phi', 'm']):
    """
    Prepares the constituent data for the GNN input.

    1. Filters fjc data to match the selected fat jets in final_events_df.
    2. Calculates features relative to the fat jet axis.
    3. Pads/truncates to a fixed size (max_constituents).
    """

    print(f"Preparing constituent tensor for {len(final_events_df)} events...")

    # Keep only the event_index and the specific fj_index from the main df
    # Also grab jet kinematics for relative feature calculation
    event_jet_map = final_events_df[['event_index', 'fj_index', 'fj_eta', 'fj_phi', 'fj_pt']]

    # Filter df_fjc to only constituents from the selected events
    fjc_filtered = df_fjc[df_fjc['event_index'].isin(event_jet_map['event_index'])]

    # Merge to align fjc constituents with their parent fat jet
    # This is the critical step that links fjc.ind with fj.index
    fjc_merged = fjc_filtered.merge(event_jet_map,
                                    left_on=['event_index', 'ind'],
                                    right_on=['event_index', 'fj_index'],
                                    how='inner')

    # --- Feature Engineering ---
    # Calculate features relative to the fat jet axis
    fjc_merged['rel_eta'] = fjc_merged['eta'] - fjc_merged['fj_eta']
    fjc_merged['rel_phi'] = delta_phi(fjc_merged['phi'], fjc_merged['fj_phi'])
    fjc_merged['rel_pt'] = fjc_merged['pt'] / fjc_merged['fj_pt'] # pt fraction
    fjc_merged['delta_r'] = np.sqrt(fjc_merged['rel_eta']**2 + fjc_merged['rel_phi']**2)

    # Define the 4 features for the GNN input, matching the example
    final_constituent_features = ['rel_pt', 'rel_eta', 'rel_phi', 'delta_r']

    # Sort constituents by pT (descending) within each event
    fjc_merged = fjc_merged.sort_values(by=['event_index', 'pt'], ascending=[True, False])

    # Group by event and collect features into a list
    grouped = fjc_merged.groupby('event_index')

    # Create the final padded/truncated tensor
    num_events = len(event_jet_map)
    num_features = len(final_constituent_features)

    # Initialize with zeros
    constituent_tensor = np.zeros((num_events, max_constituents, num_features), dtype=np.float32)

    # Use the event_index from event_jet_map to ensure correct order
    # We need to map the dataframe index (i) to the event_index
    event_index_to_tensor_row_map = {event_idx: i for i, event_idx in enumerate(event_jet_map['event_index'])}

    for event_idx, group in grouped:
        if event_idx in event_index_to_tensor_row_map:
            tensor_row_index = event_index_to_tensor_row_map[event_idx]

            # Get features and truncate
            features = group[final_constituent_features].values[:max_constituents]

            # Fill the tensor
            constituent_tensor[tensor_row_index, :len(features)] = features

    print(f"Constituent tensor created with shape: {constituent_tensor.shape}")
    return constituent_tensor

# --- Process Events (from original script) ---
NUM_FAT_JETS = 1

# Process signal and background events separately
df_events_sig = process_and_flatten_events(signal_df_lep, signal_df_j, signal_df_fj, signal_df_met)
df_events_sig['label'] = 1

# Make sure real higgs is within the fat jet
signal_df_higgs=signal_df_g[signal_df_g['id'] == 25][['event_index', 'eta', 'phi']]
df_events_sig = df_events_sig.merge(signal_df_higgs, on='event_index', how='left')
df_events_sig['hfj_dr'] = delta_r(df_events_sig['eta'], df_events_sig['fj_eta'], df_events_sig['phi'], df_events_sig['fj_phi'])
df_events_sig = df_events_sig[df_events_sig['hfj_dr'] < 1]
df_events_sig.drop(columns=['eta', 'phi', 'fj_phi', 'hfj_dr'], inplace=True)

df_events_bg_280 = process_and_flatten_events(background_280_df_lep, background_280_df_j, background_280_df_fj, background_280_df_met)
df_events_bg_280['label'] = 0
df_events_bg_280.drop(columns=['fj_phi'], inplace=True)

df_events_bg_500 = process_and_flatten_events(background_500_df_lep, background_500_df_j, background_500_df_fj, background_500_df_met)
df_events_bg_500['label'] = 0
df_events_bg_500.drop(columns=['fj_phi'], inplace=True)

df_events_bg_1000 = process_and_flatten_events(background_1000_df_lep, background_1000_df_j, background_1000_df_fj, background_1000_df_met)
df_events_bg_1000['label'] = 0
df_events_bg_1000.drop(columns=['fj_phi'], inplace=True)


"""
### Adding Event Weight
(Original script, no changes needed)
"""

# --- Calculate sizes for internal background balancing ---
size_bg_280 = len(df_events_bg_280)
size_bg_500 = len(df_events_bg_500)
size_bg_1000 = len(df_events_bg_1000)
total_bg_internal = size_bg_280 + size_bg_500 + size_bg_1000

internal_weight_280 = (total_bg_internal / 3.0) / size_bg_280 if size_bg_280 > 0 else 0
internal_weight_500 = (total_bg_internal / 3.0) / size_bg_500 if size_bg_500 > 0 else 0
internal_weight_1000 = (total_bg_internal / 3.0) / size_bg_1000 if size_bg_1000 > 0 else 0

print("--- Internal Background Weights ---")
print(f"BG 280 : Size={size_bg_280}, Internal Weight={internal_weight_280:.4f}")
print(f"BG 500 : Size={size_bg_500}, Internal Weight={internal_weight_500:.4f}")
print(f"BG 1000: Size={size_bg_1000}, Internal Weight={internal_weight_1000:.4f}")

df_events_bg_280['internal_weight'] = internal_weight_280
df_events_bg_500['internal_weight'] = internal_weight_500
df_events_bg_1000['internal_weight'] = internal_weight_1000

df_events_bg_combined = pd.concat(
    [df_events_bg_280, df_events_bg_500, df_events_bg_1000],
    ignore_index=True
)

# --- Calculate sizes for signal vs. background balancing ---
size_sig = len(df_events_sig)
size_bg_total = len(df_events_bg_combined)
total_events_final = size_sig + size_bg_total

final_weight_sig = (total_events_final / 2.0) / size_sig if size_sig > 0 else 0
total_internal_weight_sum = df_events_bg_combined['internal_weight'].sum()
final_weight_bg_multiplier = (total_events_final / 2.0) / total_internal_weight_sum if total_internal_weight_sum > 0 else 0

print("\n--- Signal vs Background Weights ---")
print(f"Signal: Size={size_sig}, Final Weight={final_weight_sig:.4f}")
print(f"Background: Raw Size={size_bg_total}, Final Multiplier={final_weight_bg_multiplier:.4f}")

# --- Apply final weights ---
df_events_sig['event_weight'] = final_weight_sig
df_events_bg_combined['event_weight'] = df_events_bg_combined['internal_weight'] * final_weight_bg_multiplier
df_events_bg_combined = df_events_bg_combined.drop(columns=['internal_weight'])

# --- Combine into the final dataset ---
df_events = pd.concat([df_events_sig, df_events_bg_combined], ignore_index=True)

# Apply padding (filling missing values)
df_events = df_events.fillna(0)

print(f"\nFinal dataset assembled with {len(df_events)} total events and final weights.")

# --- Verification ---
total_weight_sig = df_events[df_events['label'] == 1]['event_weight'].sum()
total_weight_bg = df_events[df_events['label'] == 0]['event_weight'].sum()
print(f"\nVERIFICATION: Total Sig Weight={total_weight_sig:.2f}, Total BG Weight={total_weight_bg:.2f}")


"""
## Train-Test-Val Prep (MODIFIED for GNN)
"""

# --- NEW: Generate Constituent Tensor ---
# Combine all constituent and fat jet dataframes for mapping
all_fjc = pd.concat([signal_df_fjc, background_280_df_fjc,
                     background_500_df_fjc, background_1000_df_fjc], ignore_index=True)
all_fj = pd.concat([signal_df_fj, background_280_df_fj,
                    background_500_df_fj, background_1000_df_fj], ignore_index=True)

# IMPORTANT: Ensure df_events is in a stable order before creating the tensor
# We will use this order for train/test split
df_events = df_events.sort_values(by='event_index').reset_index(drop=True)

# Generate the constituent tensor, aligned with df_events
# This tensor (X_constituents) will have the same event order as df_events
MAX_CONSTITUENTS = 50
CONSTITUENT_FEATURES = 4 # rel_pt, rel_eta, rel_phi, delta_r
X_constituents = prepare_constituent_data(all_fjc, all_fj, df_events,
                                            max_constituents=MAX_CONSTITUENTS)

# Now we can drop the index columns from the flat dataframe
df_events.drop(['event_index', 'fj_index'], axis = 1, inplace = True)

# --- MODIFIED: Split data for two inputs ---

# Separate features (X), labels (y), and weights (w)
# X_flat contains the high-level features
X_flat = df_events.drop(['label', 'event_weight'], axis=1).values
y = df_events['label'].values
w = df_events['event_weight'].values
# X_constituents is the GNN input (already created)

# Create Train, Validation, and Test splits, stratifying by y
# We must split X_flat and X_constituents together
(X_flat_train, X_flat_temp,
 X_const_train, X_const_temp,
 y_train, y_temp,
 w_train, w_temp) = train_test_split(
    X_flat, X_constituents, y, w,
    test_size=0.2, random_state=42, stratify=y
)

(X_flat_val, X_flat_test,
 X_const_val, X_const_test,
 y_val, y_test,
 w_val, w_test) = train_test_split(
    X_flat_temp, X_const_temp, y_temp, w_temp,
    test_size=0.5, random_state=42, stratify=y_temp
)

# Scale the data (only features X_flat)
scaler = StandardScaler()
X_flat_train_scaled = scaler.fit_transform(X_flat_train)
X_flat_val_scaled = scaler.transform(X_flat_val)
X_flat_test_scaled = scaler.transform(X_flat_test)

print("Data splits created and scaled (flat features only).")

# Set up device (use GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Convert numpy arrays to PyTorch tensors
# --- Flat Features ---
X_flat_train_tensor = torch.tensor(X_flat_train_scaled, dtype=torch.float32).to(device)
X_flat_val_tensor = torch.tensor(X_flat_val_scaled, dtype=torch.float32).to(device)
X_flat_test_tensor = torch.tensor(X_flat_test_scaled, dtype=torch.float32).to(device)

# --- Constituent Features (No scaling needed) ---
X_const_train_tensor = torch.tensor(X_const_train, dtype=torch.float32).to(device)
X_const_val_tensor = torch.tensor(X_const_val, dtype=torch.float32).to(device)
X_const_test_tensor = torch.tensor(X_const_test, dtype=torch.float32).to(device)

# --- Labels and Weights ---
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1).to(device)
w_train_tensor = torch.tensor(w_train, dtype=torch.float32).view(-1, 1).to(device)

X_val_tensor = torch.tensor(X_flat_val_scaled, dtype=torch.float32).to(device) # Keep for original code
y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1).to(device)
w_val_tensor = torch.tensor(w_val, dtype=torch.float32).view(-1, 1).to(device)

X_test_tensor = torch.tensor(X_flat_test_scaled, dtype=torch.float32).to(device) # Keep for original code
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).view(-1, 1).to(device)
w_test_tensor = torch.tensor(w_test, dtype=torch.float32).view(-1, 1).to(device)


# Create TensorDatasets and DataLoaders for batching
BATCH_SIZE = 256
# Dataset now includes BOTH flat and constituent tensors
train_dataset = TensorDataset(X_flat_train_tensor, X_const_train_tensor, y_train_tensor, w_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

val_dataset = TensorDataset(X_flat_val_tensor, X_const_val_tensor, y_val_tensor, w_val_tensor)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

test_dataset = TensorDataset(X_flat_test_tensor, X_const_test_tensor, y_test_tensor, w_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print("PyTorch Tensors and DataLoaders created with two inputs (flat + constituent) and weights.")


"""
## Model & Training (Phase 2 & 4: Model Definition and Training)
"""

# --- NEW: GNN Helper Function ---
def create_interaction_matrices(particles_considered):
    """
    Creates the static RR, RRT, RS, RST matrices for the GNN.
    """
    # Defines the recieving matrix for particles
    RR = []
    for i in range(particles_considered):
        row = []
        for j in range(particles_considered * (particles_considered - 1)):
            if j in range(i * (particles_considered - 1), (i + 1) * (particles_considered - 1)):
                row.append(1.0)
            else:
                row.append(0.0)
        RR.append(row)
    RR = torch.tensor(RR, dtype=torch.float32) # (N, N*(N-1))
    RRT = RR.transpose(0, 1) # (N*(N-1), N)

    # Defines the sending matrix for particles
    RST = []
    for i in range(particles_considered):
        for j in range(particles_considered):
            row = []
            for k in range(particles_considered):
                if k == j:
                    row.append(1.0)
                else:
                    row.append(0.0)
            RST.append(row)
    rowsToRemove = []
    for i in range(particles_considered):
        rowsToRemove.append(i * (particles_considered + 1))

    RST_np = np.array(RST)
    RST_np = np.delete(RST_np, rowsToRemove, 0)
    RST = torch.tensor(RST_np, dtype=torch.float32) # (N*(N-1), N)
    RS = RST.transpose(0, 1) # (N, N*(N-1))

    return RR, RRT, RS

# --- NEW: Interaction Network (GNN) Model ---
class InteractionNetwork(nn.Module):
    def __init__(self, num_particles, input_features, gnn_output_dim):
        super(InteractionNetwork, self).__init__()
        self.num_particles = num_particles

        # --- Relational Model (f_R) ---
        # Takes (2 * input_features) -> 30
        self.relational_model = nn.Sequential(
            nn.Conv1d(in_channels=2 * input_features, out_channels=80, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=80, out_channels=50, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=50, out_channels=30, kernel_size=1),
            nn.ReLU(),
            nn.BatchNorm1d(30) # Epp
        )

        # --- Object Model (f_O) ---
        # Takes (input_features + 30) -> gnn_output_dim
        self.object_model = nn.Sequential(
            nn.Conv1d(in_channels=input_features + 30, out_channels=80, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=80, out_channels=50, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(in_channels=50, out_channels=gnn_output_dim, kernel_size=1),
            nn.ReLU() # O
        )

        # Create and register matrices as non-trainable buffers
        RR, RRT, RS = create_interaction_matrices(num_particles)
        self.register_buffer('RR', RR)
        self.register_buffer('RRT', RRT)
        self.register_buffer('RS', RS)

    def forward(self, x):
        # x shape: (Batch, N, P) where N=num_particles, P=input_features

        # 1. Create relation features
        x_t = x.permute(0, 2, 1) # (B, P, N)

        # torch.matmul broadcasts RR and RS to the batch dimension
        XdotRR_t = torch.matmul(x_t, self.RR) # (B, P, N*(N-1))
        XdotRS_t = torch.matmul(x_t, self.RS) # (B, P, N*(N-1))

        XdotRR = XdotRR_t.permute(0, 2, 1) # (B, N*(N-1), P)
        XdotRS = XdotRS_t.permute(0, 2, 1) # (B, N*(N-1), P)

        Bpp = torch.cat((XdotRR, XdotRS), dim=2) # (B, N*(N-1), 2*P)

        # 2. Apply Relational Model
        Bpp_t = Bpp.permute(0, 2, 1) # (B, 2*P, N*(N-1))
        Epp_t = self.relational_model(Bpp_t) # (B, 30, N*(N-1))

        # 3. Aggregate effects
        EppBar_t = torch.matmul(Epp_t, self.RRT) # (B, 30, N)
        EppBar = EppBar_t.permute(0, 2, 1) # (B, N, 30)

        # 4. Create object features
        C = torch.cat((x, EppBar), dim=2) # (B, N, P + 30)

        # 5. Apply Object Model
        C_t = C.permute(0, 2, 1) # (B, P + 30, N)
        O_t = self.object_model(C_t) # (B, gnn_output_dim, N)

        # 6. Readout (Global Sum)
        OBar = torch.sum(O_t, dim=2) # (B, gnn_output_dim)

        return OBar

# --- NEW: Combined DNN + GNN Model ---
class CombinedModel(nn.Module):
    def __init__(self, flat_input_size, gnn_output_dim, num_particles, constituent_features):
        super(CombinedModel, self).__init__()

        # --- GNN Branch ---
        self.gnn = InteractionNetwork(
            num_particles=num_particles,
            input_features=constituent_features,
            gnn_output_dim=gnn_output_dim
        )

        # --- Flat DNN Branch (Original DNN) ---
        combined_input_size = flat_input_size + gnn_output_dim

        self.dnn = nn.Sequential(
            nn.Linear(combined_input_size, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.15),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.15),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x_flat, x_constituents):
        # Process constituents through the GNN
        gnn_out = self.gnn(x_constituents) # (B, gnn_output_dim)

        # Concatenate GNN output with flat features
        combined_features = torch.cat((x_flat, gnn_out), dim=1)

        # Pass combined vector to the original DNN
        return self.dnn(combined_features)


# --- Instantiate the NEW model ---
flat_features_dim = X_flat_train.shape[1]
gnn_output_dim = 24 # From GNN example

model = CombinedModel(
    flat_input_size=flat_features_dim,
    gnn_output_dim=gnn_output_dim,
    num_particles=MAX_CONSTITUENTS,
    constituent_features=CONSTITUENT_FEATURES
).to(device)

print(model)

# Define loss function and optimizer
loss_fn = nn.BCELoss(reduction='none')  # Use reduction='none' to get loss per element
optimizer = optim.Adam(model.parameters(), lr=0.001)

# --- Training Parameters ---
N_EPOCHS = 100
PATIENCE = 5
best_val_loss = float('inf')
patience_counter = 0

print("--- Starting Training (with GNN + DNN) ---")
for epoch in range(N_EPOCHS):
    # --- Training Phase (MODIFIED) ---
    model.train()
    train_loss = 0.0
    # Unpack the new batch format
    for inputs_flat, inputs_const, labels, weights in train_loader:
        optimizer.zero_grad()
        # Pass BOTH inputs to the model
        outputs = model(inputs_flat, inputs_const)
        loss = loss_fn(outputs, labels)
        weighted_loss = (loss * weights).mean()
        weighted_loss.backward()
        optimizer.step()
        train_loss += weighted_loss.item() * inputs_flat.size(0)

    # --- Validation Phase (MODIFIED) ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        # Unpack the new batch format
        for inputs_flat, inputs_const, labels, weights in val_loader:
            # Pass BOTH inputs to the model
            outputs = model(inputs_flat, inputs_const)
            loss = loss_fn(outputs, labels)
            weighted_loss = (loss * weights).mean()
            val_loss += weighted_loss.item() * inputs_flat.size(0)

    avg_train_loss = train_loss / len(train_loader.dataset)
    avg_val_loss = val_loss / len(val_loader.dataset)

    print(f"Epoch {epoch+1}/{N_EPOCHS} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")

    # --- Model Checkpoint & Early Stopping ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        patience_counter = 0
        print(f"  -> Validation loss improved. Saving model to 'best_model.pth'")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"  -> Early stopping triggered after {PATIENCE} epochs of no improvement.")
            break

print("\n--- Training Complete ---")


"""
## Evaluate (MODIFIED)
"""

def evaluate_model(model, data_loader, device, class_names=['Background', 'Signal']):
    model.eval()
    all_labels = []
    all_probas = []
    all_weights = []

    with torch.no_grad():
        # Unpack the new batch format
        for inputs_flat, inputs_const, labels, weights in data_loader:
            # Pass BOTH inputs to the model
            outputs = model(inputs_flat, inputs_const)
            all_labels.extend(labels.cpu().numpy())
            all_probas.extend(outputs.cpu().numpy())
            all_weights.extend(weights.cpu().numpy())


    y_test = np.array(all_labels)
    y_pred_proba = np.array(all_probas)
    y_pred_binary = (y_pred_proba > 0.5).astype(int)
    sample_weight = np.array(all_weights).flatten()


    # 1. Performance Statistics
    print("\n" + "="*60 + "\nFINAL PERFORMANCE ON TEST SET (with event weights)\n" + "="*60)
    print(classification_report(y_test, y_pred_binary, target_names=class_names, sample_weight=sample_weight))

    # 2. ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba, sample_weight=sample_weight)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(18, 5))
    plt.subplot(1, 3, 1)
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve (Weighted)'); plt.legend(loc="lower right"); plt.grid(True)

    # 3. Signal Efficiency vs. Background Rejection
    background_rejection = np.where(fpr > 0, 1.0 / fpr, np.inf)
    plt.subplot(1, 3, 2)
    plt.plot(tpr, background_rejection, color='darkgreen', lw=2) # Removed +1e-7, adjust if needed
    plt.yscale('log'); plt.xlabel('Signal Efficiency (Weighted TPR)'); plt.ylabel('Background Rejection (Weighted 1 / FPR)'); plt.title('Efficiency vs. Rejection (Weighted)'); plt.grid(True, which="both", ls="--")

    # 4. Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_binary, sample_weight=sample_weight)
    plt.subplot(1, 3, 3)
    sns.heatmap(cm, annot=True, fmt='.1f', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix (Weighted)'); plt.ylabel('True Label'); plt.xlabel('Predicted Label')

    plt.tight_layout()
    plt.show()

# --- Load the best model and evaluate (MODIFIED) ---
# Need to know the input sizes to instantiate the model
best_model = CombinedModel(
    flat_input_size=flat_features_dim,
    gnn_output_dim=gnn_output_dim,
    num_particles=MAX_CONSTITUENTS,
    constituent_features=CONSTITUENT_FEATURES
).to(device)

best_model.load_state_dict(torch.load('best_model.pth'))
print("\nBest model loaded from 'best_model.pth' for final evaluation.")

evaluate_model(best_model, test_loader, device)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.4/392.4 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.0/908.0 kB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 655.9/655.9 kB 54.0 MB/s eta 0:00:00
